# ML4, Текстовая ЛР

В данной ЛР мы собираем данные (issue из репо `kubernetes`) и учим модель классифицировать проблему- для каждой определяем один из 4 классов:
- **BUG**
- **FEATURE**
- **DOCS**
- **SUPPORT**

Разбил задачу на 3 этапа, которые я разнес по 3 файлам:

1. Сбор и разметка данных (`parser.py` и 4 файла БД)
2. Построение датасета для обучения (`dataset.py` и `isues.csv`)
3. Обучение и оценка модели (`train.py`, модель и графики)

In [ ]:
import pandas as pd

df = pd.read_csv("out/issues.csv")
df.head()

## 1. Сбор данных

Данные собирались из репозитория **kubernetes/kubernetes** через скрапинг GitHub

1. Для каждого target задаются группы label `kind/*`:

   - BUG: `kind/bug`, `kind/regression`, `kind/failing-test`, `kind/flake`
   - FEATURE: `kind/feature`, `kind/api-change`, `kind/deprecation`
   - DOCS: `kind/documentation`
   - SUPPORT: `kind/support`

2. Для каждого label строится поисковый запрос вида:

   `is:issue state:open/closed sort:created-desc label:kind/`

3. Из списка issues парсятся:
   - номер issue (`issue_id`)
   - ссылка
   - заголовок
   - полный список лэйблов

4. Для каждой issue скачивается страница, откуда извлекаются:
   - текст основного описания
   - статус
   - дата создания - не всегда правильно парсилась, иногда просто отсутствовала в скачанном DOM по до сих пор непонятным для меня причинам

5. Из заголовка и тела собирается финальное текстовое поле:
   `[TITLE] заголовок [BODY] тело проблемы`

6. Все данные сохраняются в SQLite-таблицу `issues` с полями:
   `repo, issue_id, url, target, title_raw, body_raw, text_for_model, labels_raw, created_at, state`
   При повторном запуске upsert по `(repo, issue_id)`, чтобы не дублировать записи

7.  Для каждого класса используется отдельная база:
   `issues_bug.db`, `issues_feature.db`, `issues_docs.db`, `issues_support.db`


## 2. Построение датасета для обучения (`dataset.py`)

1. Читаем все записи из 4 БД
2. Фильтруем строки с пустым `text_for_model`
3. Приводим поле `target` к одному из 4 целевых классов
4. Для каждой строки формирует запись:

   - `issue_id`, `repo`, `url`
   - `text` - очищенный `text_for_model` (заголовок + тело)
   - `label` - целочисленный ID класса
   - `label_name` - строковый класс (`BUG` / `FEATURE` / `DOCS` / `SUPPORT`)
   - `created_at`, `state`

5. Перемешиваем записи, делим на выборки:
    **80%**/ **20%**, задан сид 42

Итоговый датасет сохраняется в файл `out/issues.csv`


In [ ]:
df = pd.read_csv("out/issues.csv")

print("Размер датасета: ", df.shape, '\n')
print("Распределение классов по всей выборке: ")
print(df["label_name"].value_counts(), '\n')

print("Распределение по split :")
print(df["split"].value_counts(), '\n')

print("Распределение классов в train: ")
print(df[df["split"] == "train"]["label_name"].value_counts(), '\n')

print("Распределение классов в test: ")
print(df[df["split"] == "test"]["label_name"].value_counts())

## 3. Токенизация: BPE-модель `youtokentome`

- Из всех текстов train+test формируется корпус `bpe_corpus.txt`


- размер словаря- 16000
- специальные ID:
    - `pad_id = 0`
    - `unk_id = 1`
    - `bos_id = 2`
    - `eos_id = 3`
Итоговая модель сохраняется в файл `bpe_issues.model`

## 4. Архитектура модели и обучение

1. **Embedding-слой**
   - `nn.Embedding(num_embeddings=vocab_size, embedding_dim=128, padding_idx=0)`
   - Преобразует ID токенов в векторы размерности 128

2. **Bi-LSTM**
   - `nn.LSTM(input_size=128, hidden_size=256, num_layers=1, batch_first=True, bidirectional=True)`
   - Обрабатывает последовательность токенов слева направо и справа налево, на выход финальные hidden-состояния обоих направлений конкатенируются в вектор размерности 2 * 256 = 512

3. **Полносвязная «голова»**
   - Dropout `0.3` для регуляризации
   - Линейный слой `512 → 128` + `ReLU`
   - Ещё один Dropout `0.3`
   - Финальный линейный слой `128 → 4`, число классов

4. **Функция потерь и оптимизатор**
   - `CrossEntropy Loss`, оптимизатор `Adam` c `learning_rate = 5e-4`
   - Размер батча 64

5. **Схема обучения**
   - Модель обучается 6 эпох
   - После каждой эпохи:
     - avg loss и accuracy на train + loss и accuracy на test

Файл с весами модели сохраняется в `out/issues_ml_4_model.pt`

## 5. Результаты обучения

Итоговые метрики на тестовой выборке после 4 эпох:
- **Test loss ~ 0.85**
- **Test accuracy ~ 0.65**

Результат по классам:

- **BUG**
  - Модель находит почти все баги, но часть других типов задач ошибочно помечается как BUG

- **FEATURE**
  - Сбалансированный класс- хороший precision и recall

- **DOCS**
  - Модель иногда путает документационные задачи с BUG или FEATURE

- **SUPPORT**
  - Значительная часть support-issues уходит в BUG или DOCS, связано с похожестью описания проблемы других классов + с меньшим числом примеров для таргета в целом


## 6. Графики обучения

![Графики обучения (loss и accuracy)](graphs/training_stats.png)


- **Train Loss** на train и test убывает до 6 эпохи, пееробучения нет
- **Val loss**:
  - резко падает с ~1.3 до ~0.9 к 3-й эпохе
  - к 5-6 эпохе выходит на плато + в отдельном прогоне до 8 эпох растет после 6 (начало переобучения)
- **Val Accuracy** растет до 3 эпохи, далее колеблется 0.65-0.68

Поэтому количество эпох 6


## 7. Матрица ошибок
![Матрица ошибок](graphs/confusion_matrix.png)



- Практически все **BUG** и **FEATURE** классифицируются верно
- Часть **DOCS** и **SUPPORT** ошибочно уходят в **BUG**, связано с отсутствием строгих правил написания проблемы и разграничения лэйблов, а также с похожей на **BUG** и **FEATURE** семантикой и меньшим количеством примеров в датасете
